<a href="https://colab.research.google.com/github/Anemodude/Air-quality-index-report-system-/blob/main/M602_Computer_programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json

#request
class AirQualityApi:
    def __init__(self, latitude, longitude):
        self.latitude = latitude
        self.longitude = longitude
        self.base_url ="https://air-quality-api.open-meteo.com/v1/air-quality"

    def air_quality_fetch(self):
        params = {
            "latitude": self.latitude,
            "longitude": self.longitude,
            "hourly": ",".join([
                "pm10",
                "pm2_5",
                "carbon_monoxide",
                "nitrogen_dioxide",
                "sulphur_dioxide",
                "ozone",
                "dust",
                "uv_index"
            ])
        }

        try:
            response = requests.get(self.base_url, params=params)
            response.raise_for_status()
            data = response.json()

            with open("air_quality_raw.json", "w") as f:
                json.dump(data, f, indent=4)

            return data

        except requests.exceptions.RequestException as e:
            print(f"API Error: {e}")
            return None


In [ ]:
import csv
class AirQualityProcessing:
    def __init__(self, data):
        self.data = data

    def get_condition(self,pm10):
      if pm10 is None:
        return "Unknown"
      if pm10 <0:
        return "Invalid"
      if pm10 <= 20:
        return "Good"
      elif pm10 <= 40:
        return "Fair"
      elif pm10 <= 50:
        return "Moderate"
      elif pm10 <= 100:
        return "Poor"
      elif pm10 <= 150:
        return "Very Poor"
      else:
        return "Extremely Poor"

    def save_csv(self, filename="air_quality_index_report.csv"):
        try:
            hourly = self.data["hourly"]

            date=[]
            time=[]

            for t in hourly["time"]:
                date_part, time_part = t.split("T")
                date.append(date_part)
                time.append(time_part)

            rows=[]

            for i in range(len(hourly["time"])):
                pm10 = hourly["pm10"][i]
                condition = self.get_condition(pm10)

                rows.append([
                    date[i],
                    time[i],
                    pm10,
                    hourly["pm2_5"][i],
                    hourly["carbon_monoxide"][i],
                    hourly["nitrogen_dioxide"][i],
                    hourly["sulphur_dioxide"][i],
                    hourly["ozone"][i],
                    hourly["dust"][i],
                    hourly["uv_index"][i],
                    condition
                ])

            with open(filename, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    "date",
                    "time",
                    "pm10",
                    "pm2_5",
                    "carbon_monoxide",
                    "nitrogen_dioxide",
                    "sulphur_dioxide",
                    "ozone",
                    "dust",
                    "uv_index",
                    "condition"
                ])
                writer.writerows(rows)

        except Exception as e:
            print(f"CSV Error: {e}")


In [ ]:
class City:
    def __init__(self, filename="city.txt"):
        self.filename = filename

    def add_city(self, city):
        try:
            with open(self.filename, "a") as f:
                f.write(city + "\n")
        except Exception as e:
            print(f"File Error: {e}")

    def load_city(self):
        try:
            with open(self.filename, "r") as f:
                return [line.strip() for line in f.readlines()]
        except FileNotFoundError:
            return []


In [ ]:
# City coordinates in our case it was (Potsdam coordinates)
Latitude = 52.3989
Longitude = 13.0657

def main():
    print("Fetching air quality data for Potsdam...")

    api = AirQualityApi(Latitude, Longitude)
    data = api.air_quality_fetch()

    if data:
        processor = AirQualityProcessing(data)
        processor.save_csv()


        Air_index = City()
        Air_index.add_city("Potsdam")

        print("Air quality report generated successfully.")

if __name__ == "__main__":
    main()


Fetching air quality data for Potsdam...
Air quality report generated successfully.
